# 🔬 Phase 3 – MLflow Experiment Tracking
**Goal:** Version every experiment so results are reproducible and comparable.

Topics:
- Logging parameters, metrics, and artifacts
- Comparing runs across experiments
- Registering and versioning models
- Optuna hyperparameter search with MLflow

In [9]:
import mlflow
import mlflow.sklearn
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score
import warnings; warnings.filterwarnings('ignore')

# Connect to MLflow server
mlflow.set_tracking_uri('http://mlflow:5000')
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print('Open http://localhost:5000 in your browser to see the UI!')

MLflow tracking URI: http://mlflow:5000
Open http://localhost:5000 in your browser to see the UI!


In [10]:
# Dataset
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}, Classes: {np.unique(y)}')

Train: (142, 13), Test: (36, 13), Classes: [0 1 2]


In [ ]:
# Experiment 1: Manual model comparison
mlflow.set_experiment('Wine-Classification-Comparison')

configs = [
    ('RandomForest-100', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('RandomForest-200', RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)),
    ('GradientBoosting', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)),
]

for run_name, model in configs:
    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')
        
        # Log params
        mlflow.log_params(model.get_params())
        # Log metrics
        mlflow.log_metrics({
            'accuracy': accuracy_score(y_test, y_pred),
            'f1_macro': f1_score(y_test, y_pred, average='macro'),
            'cv_f1_mean': cv_scores.mean(),
            'cv_f1_std': cv_scores.std(),
        })
        # Log model
        mlflow.sklearn.log_model(model, 'model')
        print(f'{run_name}: acc={accuracy_score(y_test, y_pred):.4f}, f1={f1_score(y_test, y_pred, average="macro"):.4f}')

print('\nAll runs logged! Check MLflow UI at http://localhost:5000')

RandomForest-100: acc=1.0000, f1=1.0000
RandomForest-200: acc=1.0000, f1=1.0000


In [ ]:
# Experiment 2: Optuna HPO logged to MLflow
mlflow.set_experiment('Wine-HPO-Optuna')

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    }
    with mlflow.start_run(run_name=f'trial-{trial.number}', nested=True):
        model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
        score = cross_val_score(model, X_train, y_train, cv=3, scoring='f1_macro').mean()
        mlflow.log_params(params)
        mlflow.log_metric('cv_f1_macro', score)
    return score

with mlflow.start_run(run_name='optuna-study'):
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=15, show_progress_bar=True)
    
    print(f'\nBest trial: {study.best_trial.number}')
    print(f'Best F1 (CV): {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')
    mlflow.log_params(study.best_params)
    mlflow.log_metric('best_cv_f1', study.best_value)

In [ ]:
# Visualize HPO results
trials_df = study.trials_dataframe()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(trials_df['number'], trials_df['value'], 'o-', color='#3498db')
axes[0].axhline(y=study.best_value, color='red', linestyle='--', label=f'Best: {study.best_value:.4f}')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('CV F1 Score')
axes[0].set_title('Optuna: HPO Convergence', fontweight='bold')
axes[0].legend()

param_importance = optuna.importance.get_param_importances(study)
axes[1].barh(list(param_importance.keys()), list(param_importance.values()), color='#2ecc71')
axes[1].set_title('Hyperparameter Importance', fontweight='bold')
axes[1].set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../data/optuna_hpo_results.png', dpi=150, bbox_inches='tight')
plt.show()

## MLflow Concepts
| Concept | Purpose |
|---------|----------|
| **Experiment** | Group of related runs (e.g., all Wine model attempts) |
| **Run** | Single training execution with params + metrics |
| **Artifact** | Saved files: model, plots, data samples |
| **Model Registry** | Version-controlled store of production models |
| **Tags** | Free-form labels (e.g., `stage=production`) |

**Exercise:** Register the best Optuna model in MLflow Model Registry and transition it to `Staging`!